# 小样本均值置信区间的模拟

> 对应代码：`MathStatsCode/code_in_notes/Chap.9/CI_small_sample.do`（本 notebook 中的 Stata 代码与 do 文件一致）
>
> 运行方式：通过 *PyStata* 在 Jupyter 中调用 Stata，安装与使用说明见 `notebook/PyStata.md`。

**实验目的**：总体为正态、σ 未知时用 t 分布构造均值的95%置信区间，模拟10000次检验区间的覆盖率。


首先启动 Stata 环境，并把工作目录切换到 `code_in_notes/Chap.9`，这样 Stata 中的相对路径（如 `../datasets/...`）与同名 do 文件完全一致，图片也会输出到该目录。


In [ ]:
# 配置 PyStata：启动 Stata 并注册 %%stata 魔法命令
import os
os.chdir("../../code_in_notes/Chap.9")

import stata_setup
stata_setup.config("/opt/Stata", "mp")

本 notebook 通过模拟验证**小样本均值置信区间的覆盖率**，同时示范 Stata 编程的多个要点：
- `set seed 97` 固定种子；`cap program drop CIt`（`cap` 确保程序不存在时也不报错）后定义程序 `CIt`；
- `syntax [, obs(integer 10) mu(real 0) sigma(real 1) confident_level(real 0.95)]` 声明可选参数及类型；
- 程序内部：`drop _all`、`set obs \`obs'` 生成样本 `x=σ·rnormal()+μ`，求均值与标准差后按 **t 分布**构造置信区间：x̄ ± t_{α/2}(n-1)·s/√n，其中临界值 `invttail(df, alpha/2)` 是 t 分布右尾概率 α/2 对应的分位数（如 α=0.05 时为 t_{0.025}(9)=2.262）；
- 返回三个标量：区间下限、上限与**是否覆盖真值** `include_true`（区间包含 μ 输出1，否则0）；
- `simulate it=r(include_true), reps(10000): CIt, obs(10) mu(5) sigma(10)` 把整个实验重复10000次、每次样本量10，生成10000个0/1结果；`su it` 的均值就是置信区间的**实际覆盖率**——理论上应接近95%。这套“定义程序→simulate→统计覆盖频率”的模式是检验区间估计有效性的标准做法。


In [ ]:
%%stata
clear
set seed 97
set more off
cap program drop CIt
program define CIt, rclass
	syntax [, obs(integer 10) mu(real 0) sigma(real 1) confident_level(real 0.95)]
	quietly{
		drop _all
		set obs `obs'
		tempvar x 
		tempname lb ub df alpha
		local `alpha'=1-`confident_level'
		local `df'=`obs'-1
		gen `x'=`sigma'*rnormal()+`mu'
		su `x'
		local `lb'=r(mean)-invttail(``df'',``alpha''/2)*r(sd)/sqrt(`obs')
		local `ub'=r(mean)+invttail(``df'',``alpha''/2)*r(sd)/sqrt(`obs')
		return scalar lb=``lb''
		return scalar ub=``ub''
		return scalar include_true=(``lb''<=`mu' & ``ub''>=`mu')
	}
end

simulate it=r(include_true), reps(10000): CIt, obs(10) mu(5) sigma(10)
su
